In [1]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ==========================================
# 1. Load Saved Model & Tokenizer Mapping
# ==========================================
MODEL_PATH = 'D:/VishnuMK/Projects/Email/review/best_cnn_lstm_phishing_model.keras'        # or 'best_cnn_lstm_phishing_model.keras'
TOKENIZER_PATH = 'D:/VishnuMK/Projects/Email/review/tokenizer_word_index.json'

# Load the trained model
model = tf.keras.models.load_model(MODEL_PATH)

# Load the character-to-integer dictionary saved during preprocessing
with open(TOKENIZER_PATH, 'r') as f:
    word_index = json.load(f)


In [2]:

# ==========================================
# 2. Preprocessing Function for Raw URLs
# ==========================================
def preprocess_custom_url(url, word_index, max_len=200):
    """
    Converts a single raw URL string into a padded character sequence 
    matching the trained model's input format.
    """
    url_lower = url.lower().strip()
    
    # Map characters to integer IDs using the loaded tokenizer dictionary
    # Unknown characters default to 1 (the <UNK> token ID)
    unk_id = word_index.get('<UNK>', 1)
    sequence = [word_index.get(char, unk_id) for char in url_lower]
    
    # Pad or truncate sequence to 200 length
    padded_sequence = pad_sequences([sequence], maxlen=max_len, padding='post', truncating='post')
    return padded_sequence


In [3]:

# ==========================================
# 3. Prediction Function
# ==========================================
def predict_url(url):
    preprocessed_input = preprocess_custom_url(url, word_index)
    
    # Output probability from sigmoid layer (0.0 to 1.0)
    prob = model.predict(preprocessed_input, verbose=0)[0][0]
    
    # Classification decision
    label = "PHISHING ⚠️" if prob > 0.5 else "BENIGN ✅"
    confidence = prob if prob > 0.5 else (1 - prob)
    
    print("-" * 60)
    print(f"URL: {url}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence * 100:.2f}% (Phishing Probability: {prob:.4f})")
    print("-" * 60)



In [4]:
# ==========================================
# 4. Test Custom URLs
# ==========================================
test_urls = [
    "https://www.google.com",
    "http://paypal.com-verification-account.login.ru/update",
    "https://github.com/tensorflow/tensorflow",
    "http://amazon.com.security-check-portal.com/auth"
]

for url in test_urls:
    predict_url(url)

------------------------------------------------------------
URL: https://www.google.com
Prediction: PHISHING ⚠️
Confidence: 99.59% (Phishing Probability: 0.9959)
------------------------------------------------------------
------------------------------------------------------------
URL: http://paypal.com-verification-account.login.ru/update
Prediction: PHISHING ⚠️
Confidence: 99.57% (Phishing Probability: 0.9957)
------------------------------------------------------------
------------------------------------------------------------
URL: https://github.com/tensorflow/tensorflow
Prediction: PHISHING ⚠️
Confidence: 99.53% (Phishing Probability: 0.9953)
------------------------------------------------------------
------------------------------------------------------------
URL: http://amazon.com.security-check-portal.com/auth
Prediction: PHISHING ⚠️
Confidence: 99.56% (Phishing Probability: 0.9956)
------------------------------------------------------------
